In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

source_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_trips"

bronze_incremental_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

batch_log_table = "learn_get_started_de_palvikrant235.taxi_etl_project.pipeline_batch_log"

batch_id = "BATCH_005"

test_input_table = "learn_get_started_de_palvikrant235.taxi_etl_project.test_incoming_batch"

df_batch = spark.table(test_input_table)

print("Incoming Records:", df_batch.count())

Incoming Records: 10


In [0]:
from pyspark.sql import functions as F

source_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_trips"

df_source = spark.table(source_table)

df_batched = df_source.withColumn(
    "batch_number",
    F.pmod(F.xxhash64("id"), F.lit(3))
)

In [0]:
batch_log_table = "learn_get_started_de_palvikrant235.taxi_etl_project.pipeline_batch_log"

bronze_incremental_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

for batch_number in range(3):

    batch_id = f"BATCH_{batch_number + 1:03d}"

    already_processed = (
        spark.table(batch_log_table)
        .filter(
            (F.col("batch_id") == batch_id) &
            (F.col("batch_status") == "SUCCESS")
        )
        .limit(1)
        .count() > 0
    )

    if already_processed:
        print(f"{batch_id} already processed. Skipping.")
        continue

    df_batch = (
        df_batched
        .filter(F.col("batch_number") == batch_number)
        .drop("batch_number")
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("source_file", F.lit("train.csv"))
        .withColumn("source_system", F.lit("KAGGLE"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

    df_batch.write.format("delta").mode("append").saveAsTable(
        bronze_incremental_table
    )

    batch_count = df_batch.count()

    log_df = spark.createDataFrame(
        [(batch_id, "SUCCESS", batch_count)],
        ["batch_id", "batch_status", "records_processed"]
    ).withColumn("processed_timestamp", F.current_timestamp())

    log_df.write.format("delta").mode("append").saveAsTable(
        batch_log_table
    )

    print(f"{batch_id} completed: {batch_count} records")

BATCH_001 already processed. Skipping.
BATCH_002 already processed. Skipping.
BATCH_003 already processed. Skipping.


In [0]:
print("Incremental Bronze Count:",
      spark.table("learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental").count())

Incremental Bronze Count: 1458654


In [0]:
batch_id = "BATCH_005"

already_processed = (
    spark.table(batch_log_table)
    .filter(
        (F.col("batch_id") == batch_id) &
        (F.col("batch_status") == "SUCCESS")
    )
    .limit(1)
    .count() > 0
)

if already_processed:
    print(f"{batch_id} already processed. Skipping.")

else:
    test_input_table = "learn_get_started_de_palvikrant235.taxi_etl_project.test_incoming_batch"
    df_batch = spark.table(test_input_table)

    batch_count = df_batch.count()

    (
        df_batch.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_incremental_table)
    )

    print(f"{batch_id} ingested successfully: {batch_count} records")

BATCH_005 already processed. Skipping.


In [0]:
if not already_processed:

    log_df = spark.createDataFrame(
        [(batch_id, "SUCCESS", batch_count)],
        ["batch_id", "batch_status", "records_processed"]
    ).withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )

    log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(batch_log_table)

    print(f"{batch_id} successfully registered in batch log.")

else:
    print("No new batch logging required.")

No new batch logging required.


In [0]:
from pyspark.sql import functions as F

test_input_table = "learn_get_started_de_palvikrant235.taxi_etl_project.test_incoming_batch"

df_batch = spark.table(test_input_table)

print("DataFrame loaded successfully")
print("Records:", df_batch.count())

DataFrame loaded successfully
Records: 10


In [0]:
print("Incoming Batch Count:", df_batch.count())

print("Null IDs:",
      df_batch.filter(F.col("id").isNull()).count())

print("Duplicate IDs:",
      df_batch.count() - df_batch.select("id").distinct().count())

if df_batch.count() != 10:
    raise Exception("Unexpected batch record count")

if df_batch.filter(F.col("id").isNull()).count() > 0:
    raise Exception("Null IDs found")

if df_batch.count() != df_batch.select("id").distinct().count():
    raise Exception("Duplicate IDs found")

print("Incoming batch validation PASSED")

Incoming Batch Count: 10
Null IDs: 0
Duplicate IDs: 0
Incoming batch validation PASSED


In [0]:
bronze_incremental_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

print(bronze_incremental_table)

learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental


In [0]:
from pyspark.sql import functions as F

df_batch = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema("""
        id STRING,
        vendor_id INT,
        pickup_datetime TIMESTAMP,
        dropoff_datetime TIMESTAMP,
        passenger_count INT,
        pickup_longitude DOUBLE,
        pickup_latitude DOUBLE,
        dropoff_longitude DOUBLE,
        dropoff_latitude DOUBLE,
        store_and_fwd_flag STRING,
        trip_duration INT
    """)
    .load("/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/raw_files/auto_loader_test_input/")
    .withColumn("batch_id", F.lit("AUTO_TEST_001"))
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("source_system", F.lit("KAGGLE"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

print("Incoming Auto Loader Test Records:", df_batch.count())

Incoming Auto Loader Test Records: 2


In [0]:
df_bronze_existing = spark.table(bronze_incremental_table)

df_new_to_bronze = (
    df_batch.alias("source")
    .join(
        df_bronze_existing.select("id").alias("target"),
        on="id",
        how="left_anti"
    )
)

print("Incoming Records:", df_batch.count())
print("New Records:", df_new_to_bronze.count())
print("Already Existing:", df_batch.count() - df_new_to_bronze.count())

Incoming Records: 10
New Records: 0
Already Existing: 10


In [0]:
from delta.tables import DeltaTable

bronze_target = DeltaTable.forName(
    spark,
    bronze_incremental_table
)

(
    bronze_target.alias("target")
    .merge(
        df_new_to_bronze.alias("source"),
        "target.id = source.id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Bronze MERGE completed successfully")

Bronze MERGE completed successfully


In [0]:
print(
    "Bronze Incremental Count:",
    spark.table(bronze_incremental_table).count()
)

Bronze Incremental Count: 1458654


In [0]:
print(
    "Bronze Total Records:",
    spark.table(bronze_incremental_table).count()
)

print(
    "BATCH_005 Records:",
    spark.table(bronze_incremental_table)
    .filter(F.col("batch_id") == "BATCH_005")
    .count()
)

Bronze Total Records: 1458654
BATCH_005 Records: 10


In [0]:
print("Bronze Total Records:",
      spark.table(bronze_incremental_table).count())

print("BATCH_005 Records:",
      spark.table(bronze_incremental_table)
      .filter(F.col("batch_id") == "BATCH_005")
      .count())

Bronze Total Records: 1458654
BATCH_005 Records: 10


In [0]:
batch_log_table = "learn_get_started_de_palvikrant235.taxi_etl_project.pipeline_batch_log"

batch_id = "BATCH_005"

existing_log = (
    spark.table(batch_log_table)
    .filter(F.col("batch_id") == batch_id)
    .count()
)

if existing_log == 0:

    log_df = spark.createDataFrame(
        [("BATCH_005", "SUCCESS", 10)],
        ["batch_id", "batch_status", "records_processed"]
    ).withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )

    log_df.write.format("delta").mode("append").saveAsTable(
        batch_log_table
    )

    print("BATCH_005 registered successfully")

else:
    print("BATCH_005 already exists in batch log")

BATCH_005 already exists in batch log


In [0]:
display(
    spark.table(batch_log_table)
    .filter(F.col("batch_id") == "BATCH_005")
)

batch_id,batch_status,records_processed,processed_timestamp
BATCH_005,SUCCESS,10,2026-11-18T10:42:25.376Z


In [0]:
batch_id = "BATCH_005"

already_processed = (
    spark.table(batch_log_table)
    .filter(
        (F.col("batch_id") == batch_id) &
        (F.col("batch_status") == "SUCCESS")
    )
    .limit(1)
    .count() > 0
)

if already_processed:
    print(f"{batch_id} already processed. Skipping ingestion.")
else:
    print(f"{batch_id} is new. Processing required.")

BATCH_005 already processed. Skipping ingestion.


In [0]:
print(
    "Bronze Total Records:",
    spark.table(bronze_incremental_table).count()
)

Bronze Total Records: 1458654


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

batch_id = "BATCH_005"

test_input_table = "learn_get_started_de_palvikrant235.taxi_etl_project.test_incoming_batch"

bronze_incremental_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

batch_log_table = "learn_get_started_de_palvikrant235.taxi_etl_project.pipeline_batch_log"

# Read persistent incoming data
df_batch = spark.table(test_input_table)

# Check batch log
already_processed = (
    spark.table(batch_log_table)
    .filter(
        (F.col("batch_id") == batch_id) &
        (F.col("batch_status") == "SUCCESS")
    )
    .limit(1)
    .count() > 0
)

if already_processed:
    print(f"{batch_id} already processed. Skipping.")

else:
    # Validate incoming data
    total_records = df_batch.count()
    duplicate_count = total_records - df_batch.select("id").distinct().count()
    null_count = df_batch.filter(F.col("id").isNull()).count()

    if total_records != 10 or duplicate_count > 0 or null_count > 0:
        raise Exception("Incoming batch validation failed")

    # Insert only new IDs
    delta_target = DeltaTable.forName(spark, bronze_incremental_table)

    (
        delta_target.alias("target")
        .merge(
            df_batch.alias("source"),
            "target.id = source.id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    # Register successful batch
    log_df = spark.createDataFrame(
        [(batch_id, "SUCCESS", total_records)],
        ["batch_id", "batch_status", "records_processed"]
    ).withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )

    log_df.write.format("delta").mode("append").saveAsTable(batch_log_table)

    print(f"{batch_id} processed successfully")

BATCH_005 already processed. Skipping.


In [0]:
print(
    "Bronze Total Records:",
    spark.table(bronze_incremental_table).count()
)

print(
    "BATCH_005 Records:",
    spark.table(bronze_incremental_table)
    .filter(F.col("batch_id") == "BATCH_005")
    .count()
)

Bronze Total Records: 1458654
BATCH_005 Records: 10


In [0]:
display(dbutils.fs.ls(
    "/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/raw_files/"
))

path,name,size,modificationTime
dbfs:/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/raw_files/NYC.csv,NYC.csv,200589097,1793685369000


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

source_path = "/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/raw_files/"

schema_path = "/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/checkpoints/schema"

checkpoint_path = "/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/checkpoints/bronze"

bronze_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

df_new_files = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
    .withColumn("batch_id", F.lit("AUTO_LOADER"))
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("source_system", F.lit("KAGGLE"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

def ingest_to_bronze(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    target = DeltaTable.forName(spark, bronze_table)

    (
        target.alias("target")
        .merge(
            batch_df.alias("source"),
            "target.id = source.id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_new_files.writeStream
    .foreachBatch(ingest_to_bronze)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Auto Loader completed successfully")

In [0]:
print("Bronze schema:")
spark.table(
    "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"
).printSchema()

Bronze schema:
root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F

bronze_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

test_path = "/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/raw_files/auto_loader_test_input"

raw_columns = [
    "id", "vendor_id", "pickup_datetime", "dropoff_datetime",
    "passenger_count", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "store_and_fwd_flag",
    "trip_duration"
]

df_test = (
    spark.table(bronze_table)
    .limit(2)
    .withColumn("id", F.concat(F.lit("AUTO_TEST_"), F.col("id")))
    .select(*raw_columns)
)

df_test.coalesce(1).write.mode("overwrite").option("header", "true").csv(test_path)

print("Test file created:", test_path)

Test file created: /Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/raw_files/auto_loader_test_input


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS
learn_get_started_de_palvikrant235.taxi_etl_project.checkpoints;

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

test_source = "/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/raw_files/auto_loader_test_input/"

test_schema = "/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/checkpoints/test_schema"

test_checkpoint = "/Volumes/learn_get_started_de_palvikrant235/taxi_etl_project/checkpoints/test_bronze"

bronze_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema("""
        id STRING,
        vendor_id INT,
        pickup_datetime TIMESTAMP,
        dropoff_datetime TIMESTAMP,
        passenger_count INT,
        pickup_longitude DOUBLE,
        pickup_latitude DOUBLE,
        dropoff_longitude DOUBLE,
        dropoff_latitude DOUBLE,
        store_and_fwd_flag STRING,
        trip_duration INT
    """)
    .option("cloudFiles.schemaLocation", test_schema)
    .load(test_source)
    .withColumn("batch_id", F.lit("AUTO_TEST_001"))
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("source_system", F.lit("KAGGLE"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

def save_test_batch(batch_df, batch_id):
    target = DeltaTable.forName(spark, bronze_table)

    (
        target.alias("t")
        .merge(batch_df.alias("s"), "t.id = s.id")
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_stream.writeStream
    .foreachBatch(save_test_batch)
    .option("checkpointLocation", test_checkpoint)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Auto Loader test completed!")

Auto Loader test completed!


In [0]:
print(
    "Auto Test Records:",
    spark.table(
        "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"
    )
    .filter(F.col("batch_id") == "AUTO_TEST_001")
    .count()
)

Auto Test Records: 2


In [0]:
from pyspark.sql import functions as F

bronze_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

print(
    "Auto Test Records:",
    spark.table(bronze_table)
    .filter(F.col("batch_id") == "AUTO_TEST_001")
    .count()
)

Auto Test Records: 2


In [0]:
print(
    "Auto Test Records:",
    spark.table(bronze_table)
    .filter(F.col("batch_id") == "AUTO_TEST_001")
    .count()
)

Auto Test Records: 2


In [0]:
print("Incoming Records:", df_batch.count())
print("New Records:", df_new_to_bronze.count())
print("Already Existing:", df_batch.count() - df_new_to_bronze.count())

Incoming Records: 10
New Records: 0
Already Existing: 10
